Get User Data

In [1]:
import os
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import ContentBasedRecommender

load_dotenv()

client_id = os.getenv("CLIENT_ID")

In [2]:
# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})

Users data

In [3]:
# users_data = {}
# users_scores = {}

# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})
# mal_client = MALClient(client_id)
# for user in users:
#     user_data = mal_client.get_user_data(user)
#     users_data[user] = user_data
#     scores = mal_client.get_scores(user_data)
#     users_scores[user] = scores

In [4]:
anime_data_client = AnimeDataClient(client_id)

In [5]:
anime_data = anime_data_client.get_cache()

In [6]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()
builder.svd_explained_variance

np.float64(0.4104185921487374)

Convert each anime in df to vectors

In [13]:
recommender = ContentBasedRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Bayesion Ridge Regression

In [32]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

In [33]:
import numpy as np
import pandas as pd
from sklearn.linear_model import BayesianRidge

# Keep X and y aligned: each vector is paired with that anime's user score.
rated_items = [
    (anime_id, anime_vectors[anime_id], score)
    for anime_id, score in user_scores.items()
    if anime_id in anime_vectors and score not in (None, 0, "-")
]

rated_ids = {anime_id for anime_id, _, _ in rated_items}
X_train = np.array([vec for _, vec, _ in rated_items])
y_train = np.array([score for _, _, score in rated_items], dtype=float)

model = BayesianRidge()
model.fit(X_train, y_train)

candidate_ids = [anime_id for anime_id in anime_df_scaled.index if anime_id not in rated_ids]
X_candidates = anime_df_scaled.loc[candidate_ids].to_numpy()

pred_mean, pred_std = model.predict(X_candidates, return_std=True)

# Approximate 95% confidence interval. Clip to MAL's 1-10 score range.
lower = np.clip(pred_mean - 1.96 * pred_std, 1, 10)
upper = np.clip(pred_mean + 1.96 * pred_std, 1, 10)
pred_mean = np.clip(pred_mean, 1, 10)

In [34]:
title_by_id = {int(anime["id"]): anime["title"] for anime in anime_data.values()}

recommendations = pd.DataFrame({
    "anime_id": candidate_ids,
    "title": [title_by_id.get(int(anime_id), "Unknown") for anime_id in candidate_ids],
    "predicted_score": pred_mean,
    "ci_lower": lower,
    "ci_upper": upper,
    "uncertainty": pred_std,
})

if len(y_train) < 10:
    ranking_score = 0.7 * recommendations["mean"] + 0.3 * recommendations["predicted_score"]
else:
    ranking_score = recommendations["predicted_score"] - 0.5 * recommendations["uncertainty"]

recommendations["ranking_score"] = ranking_score
recommendations = recommendations.sort_values("ranking_score", ascending=False)
recommendations.head(20)

,anime_id,title,predicted_score,ci_lower,ci_upper,uncertainty,ranking_score
4,1535,Death Note,9.044505,6.072762,10.0,1.516196,8.286408
58,36296,Hinamatsuri,8.766038,5.592720,10.0,1.619040,7.956518
360,30,Shinseiki Evangelion,8.700007,5.741949,10.0,1.509213,7.945400
353,1,Cowboy Bebop,8.650656,5.826303,10.0,1.440996,7.930158
1727,440,Shoujo Kakumei Utena,8.631409,5.725399,10.0,1.482658,7.890080
374,36649,Banana Fish,8.569516,5.470775,10.0,1.580991,7.779021
1753,53411,Buddy Daddies,8.530500,5.509026,10.0,1.541568,7.759716
248,4224,Toradora!,8.457465,5.628957,10.0,1.443116,7.735907
2,11061,Hunter x Hunter (2011),8.445799,5.605832,10.0,1.448962,7.721318
1912,2800,Candy Candy,8.435202,5.594674,10.0,1.449249,7.710577
